# Altair Dashboard Prototypes

Six reusable visual implementations over your parquet indicators.

In [ ]:

from pathlib import Path
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists():
        return cwd
    if (cwd.parent / 'data').exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = resolve_project_root()
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'


def find_parquet(*keywords: str) -> Path:
    keys = [k.lower() for k in keywords]
    for p in sorted(DATA_ROOT.rglob('*.parquet')):
        n = str(p).lower()
        if all(k in n for k in keys):
            return p
    raise FileNotFoundError(f'No parquet found for keywords={keywords}')


def load_indicator(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()
    df['value'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    df['year'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
    df = df.dropna(subset=['value', 'year'])
    df['year'] = df['year'].astype(int)
    return df


def latest_per_country(df: pd.DataFrame) -> pd.DataFrame:
    latest = int(df['year'].max())
    return df[df['year'] == latest].copy()


In [ ]:

# Representative datasets used in the six charts
paths = {
    'governance': find_parquet('governance', 'control_corruption'),
    'internet': find_parquet('internet', 'percentage_population_using_internet'),
    'trade': find_parquet('trade', 'commodity_terms_trade'),
    'rule_law_overall': find_parquet('rule of law', 'overall_score'),
    'rule_law_low': find_parquet('governance', 'rule_percentile_lower_bound'),
    'rule_law_high': find_parquet('governance', 'rule_percentile_upper_bound'),
}
paths


## 1) Country trend explorer (interactive line chart)

In [ ]:

df_gov = load_indicator(paths['governance'])
rank_latest = latest_per_country(df_gov).nlargest(20, 'value')['REF_AREA'].tolist()
base = df_gov[df_gov['REF_AREA'].isin(rank_latest)]

select_country = alt.selection_point(fields=['REF_AREA'], on='click', empty='all')

chart_trend = (
    alt.Chart(base)
    .mark_line(point=True)
    .encode(
        x=alt.X('year:Q', title='Year'),
        y=alt.Y('value:Q', title='Control of Corruption (OBS_VALUE)'),
        color=alt.condition(select_country, 'REF_AREA:N', alt.value('lightgray')),
        tooltip=['REF_AREA:N', 'year:Q', alt.Tooltip('value:Q', format='.3f')]
    )
    .add_params(select_country)
    .properties(width=900, height=350)
)
chart_trend


## 2) Latest-year choropleth map

In [ ]:

latest_gov = latest_per_country(df_gov)
latest_year = int(latest_gov['year'].max())

world_geo = alt.Data(url='https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson', format=alt.DataFormat(property='features'))

chart_map = (
    alt.Chart(world_geo)
    .mark_geoshape(stroke='white', strokeWidth=0.3)
    .transform_lookup(
        lookup='properties.ISO_A3',
        from_=alt.LookupData(latest_gov, 'REF_AREA', ['value'])
    )
    .encode(
        color=alt.Color('value:Q', title=f'Control of Corruption ({latest_year})', scale=alt.Scale(scheme='teals')),
        tooltip=[alt.Tooltip('properties.ADMIN:N', title='Country'), alt.Tooltip('properties.ISO_A3:N', title='REF_AREA'), alt.Tooltip('value:Q', format='.3f')]
    )
    .project(type='equalEarth')
    .properties(width=900, height=450)
)
chart_map


## 3) Top/Bottom ranking bars (year slider)

In [ ]:

years = sorted(df_gov['year'].unique())
year_param = alt.param(value=int(max(years)), bind=alt.binding_range(min=int(min(years)), max=int(max(years)), step=1, name='Year: '))

rank_base = (
    alt.Chart(df_gov)
    .transform_filter(alt.datum.year == year_param)
    .add_params(year_param)
)

top10 = rank_base.transform_window(rank='rank(value)', sort=[alt.SortField('value', order='descending')]).transform_filter(alt.datum.rank <= 10)
bot10 = rank_base.transform_window(rank='rank(value)', sort=[alt.SortField('value', order='ascending')]).transform_filter(alt.datum.rank <= 10)

chart_top = top10.mark_bar(color='#1f77b4').encode(y=alt.Y('REF_AREA:N', sort='-x', title='Top 10'), x=alt.X('value:Q', title='Value'), tooltip=['REF_AREA:N', 'year:Q', alt.Tooltip('value:Q', format='.3f')]).properties(width=430, height=280)
chart_bot = bot10.mark_bar(color='#d62728').encode(y=alt.Y('REF_AREA:N', sort='x', title='Bottom 10'), x=alt.X('value:Q', title='Value'), tooltip=['REF_AREA:N', 'year:Q', alt.Tooltip('value:Q', format='.3f')]).properties(width=430, height=280)

chart_top | chart_bot


## 4) Scatter with regression (internet vs corruption)

In [ ]:

df_net = load_indicator(paths['internet'])
common_year = int(min(df_gov['year'].max(), df_net['year'].max()))

x = df_net[df_net['year'] == common_year][['REF_AREA', 'value']].rename(columns={'value': 'internet_value'})
y = df_gov[df_gov['year'] == common_year][['REF_AREA', 'value']].rename(columns={'value': 'corruption_value'})
xy = x.merge(y, on='REF_AREA', how='inner')

points = alt.Chart(xy).mark_circle(size=70, opacity=0.75).encode(
    x=alt.X('internet_value:Q', title=f'Internet usage ({common_year})'),
    y=alt.Y('corruption_value:Q', title=f'Control of Corruption ({common_year})'),
    tooltip=['REF_AREA:N', alt.Tooltip('internet_value:Q', format='.2f'), alt.Tooltip('corruption_value:Q', format='.3f')]
)
line = points.transform_regression('internet_value', 'corruption_value').mark_line(color='black')
(points + line).properties(width=900, height=380)


## 5) Heatmap (country × year)

In [ ]:

df_trade = load_indicator(paths['trade'])
latest_trade = latest_per_country(df_trade)
selected = latest_trade.nlargest(35, 'value')['REF_AREA'].tolist()
heat = df_trade[df_trade['REF_AREA'].isin(selected)]

chart_heat = alt.Chart(heat).mark_rect().encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('REF_AREA:N', sort=selected, title='Country (top latest terms of trade)'),
    color=alt.Color('value:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['REF_AREA:N', 'year:Q', alt.Tooltip('value:Q', format='.2f')]
).properties(width=950, height=520)
chart_heat


## 6) Uncertainty bands (lower/upper confidence interval)

In [ ]:

low = load_indicator(paths['rule_law_low'])[['REF_AREA', 'year', 'value']].rename(columns={'value': 'low'})
high = load_indicator(paths['rule_law_high'])[['REF_AREA', 'year', 'value']].rename(columns={'value': 'high'})
band_df = low.merge(high, on=['REF_AREA', 'year'], how='inner')
band_df = band_df[band_df['high'] >= band_df['low']].copy()

country_options = sorted(band_df['REF_AREA'].unique().tolist())
country_param = alt.param(value=country_options[0], bind=alt.binding_select(options=country_options, name='REF_AREA: '))

band = alt.Chart(band_df).transform_filter(alt.datum.REF_AREA == country_param).mark_area(opacity=0.25, color='#1f77b4').encode(
    x=alt.X('year:Q', title='Year'),
    y=alt.Y('low:Q', title='Rule of Law percentile bounds'),
    y2='high:Q'
).add_params(country_param)

mid = alt.Chart(band_df).transform_filter(alt.datum.REF_AREA == country_param).transform_calculate(mid='(datum.low + datum.high) / 2').mark_line(color='#0d3b66').encode(
    x='year:Q',
    y=alt.Y('mid:Q'),
    tooltip=['REF_AREA:N', 'year:Q', alt.Tooltip('low:Q', format='.3f'), alt.Tooltip('high:Q', format='.3f'), alt.Tooltip('mid:Q', format='.3f')]
)

(band + mid).properties(width=900, height=350)


## 7) Bubble world map (size = human capital proxy, color = trade openness)

Using `wealth/human_capital.parquet` as requested for bubble size and `trade/merchandise_trade_(%_gdp).parquet` for trade openness color.


In [ ]:

trade_path = find_parquet('trade', 'merchandise_trade', 'gdp')
hc_path = find_parquet('wealth', 'human_capital')

trade_df = load_indicator(trade_path)[['REF_AREA', 'year', 'value']].rename(columns={'value': 'trade_open'})
hc_df = load_indicator(hc_path)[['REF_AREA', 'year', 'value']].rename(columns={'value': 'human_capital'})

map_df = trade_df.merge(hc_df, on=['REF_AREA', 'year'], how='inner')
map_year = int(map_df['year'].max())
map_latest = map_df[map_df['year'] == map_year].copy()

# country centroids (ISO2) + ISO2->ISO3 lookup
centroids = pd.read_csv('https://raw.githubusercontent.com/gavinr/world-countries-centroids/master/dist/countries.csv')
iso = pd.read_csv('https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv')
centroids_iso3 = centroids.merge(iso[['alpha-2', 'alpha-3']], left_on='ISO', right_on='alpha-2', how='left')
centroids_iso3 = centroids_iso3.rename(columns={'alpha-3': 'REF_AREA'})

map_plot = map_latest.merge(
    centroids_iso3[['REF_AREA', 'COUNTRY', 'longitude', 'latitude']],
    on='REF_AREA',
    how='inner'
)

world_geo = alt.Data(
    url='https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson',
    format=alt.DataFormat(property='features')
)

base_map = alt.Chart(world_geo).mark_geoshape(
    fill='whitesmoke', stroke='white', strokeWidth=0.5
).project(type='equalEarth').properties(width=950, height=460)

bubbles = alt.Chart(map_plot).mark_circle(opacity=0.85, stroke='black', strokeWidth=0.25).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.Size('human_capital:Q', title='Human capital (proxy size)', scale=alt.Scale(range=[20, 2200])),
    color=alt.Color('trade_open:Q', title=f'Trade openness % GDP ({map_year})', scale=alt.Scale(scheme='yellowgreenblue')),
    tooltip=[
        alt.Tooltip('COUNTRY:N', title='Country'),
        alt.Tooltip('REF_AREA:N', title='ISO3'),
        alt.Tooltip('trade_open:Q', title='Trade openness % GDP', format='.2f'),
        alt.Tooltip('human_capital:Q', title='Human capital', format='.2f')
    ]
)

(base_map + bubbles).properties(title=f'Bubble Map — Year {map_year}')


## 8) Radial chart (country selector, auto-pick top available metrics)


In [ ]:

metric_sources = {
    'Trade openness (% GDP)': find_parquet('trade', 'merchandise_trade', 'gdp'),
    'Human capital': find_parquet('wealth', 'human_capital'),
    'Control of corruption': find_parquet('governance', 'control_corruption'),
    'Rule of law (WJP overall)': find_parquet('rule of law', 'overall_score'),
    'Internet usage (%)': find_parquet('internet', 'percentage_population_using_internet'),
    'Urban population (%)': find_parquet('urbanization', 'urban_population_(%_total_population)'),
    'HDI': find_parquet('development', 'human_development_index'),
    'Press freedom score': find_parquet('press freedom', 'index_score'),
}

metric_frames = []
for metric_name, path in metric_sources.items():
    d = load_indicator(path)[['REF_AREA', 'year', 'value']].copy()
    d['metric'] = metric_name
    metric_frames.append(d)

long_df = pd.concat(metric_frames, ignore_index=True)

# Normalize each metric to 0-100 globally for comparability
metric_stats = long_df.groupby('metric', as_index=False)['value'].agg(vmin='min', vmax='max')
long_df = long_df.merge(metric_stats, on='metric', how='left')
long_df['score'] = ((long_df['value'] - long_df['vmin']) / (long_df['vmax'] - long_df['vmin']).replace(0, pd.NA) * 100).fillna(0)

# Latest value per country+metric (auto-pick the most available/highest-scoring metrics)
latest_idx = long_df.sort_values('year').groupby(['REF_AREA', 'metric'])['year'].idxmax()
latest = long_df.loc[latest_idx, ['REF_AREA', 'metric', 'year', 'value', 'score']].copy()

top_n = 6
latest = latest.sort_values(['REF_AREA', 'score'], ascending=[True, False])
latest['metric_rank'] = latest.groupby('REF_AREA').cumcount() + 1
radar_df = latest[latest['metric_rank'] <= top_n].copy()

# Close polygon by repeating the first metric per country
radar_df = radar_df.sort_values(['REF_AREA', 'metric_rank'])
first = radar_df.groupby('REF_AREA', as_index=False).first()
first['metric_rank'] = top_n + 1
first['metric'] = first['metric'] + ' (close)'
radar_closed = pd.concat([radar_df, first], ignore_index=True)

countries = sorted(radar_closed['REF_AREA'].unique().tolist())
country_param = alt.param(value=countries[0], bind=alt.binding_select(options=countries, name='Country (ISO3): '))

radial_line = alt.Chart(radar_closed).transform_filter(
    alt.datum.REF_AREA == country_param
).mark_line(point=True).encode(
    theta=alt.Theta('metric_rank:Q', stack=False),
    radius=alt.Radius('score:Q', scale=alt.Scale(domain=[0, 100])),
    tooltip=[
        alt.Tooltip('REF_AREA:N', title='Country'),
        alt.Tooltip('metric:N', title='Metric'),
        alt.Tooltip('value:Q', title='Raw value', format='.3f'),
        alt.Tooltip('score:Q', title='Normalized score', format='.1f'),
        alt.Tooltip('year:Q', title='Year')
    ],
    color=alt.value('#1f77b4')
).add_params(country_param).properties(width=500, height=500)

radial_labels = alt.Chart(radar_df).transform_filter(
    alt.datum.REF_AREA == country_param
).mark_text(radiusOffset=12, fontSize=10).encode(
    theta=alt.Theta('metric_rank:Q', stack=False),
    radius=alt.value(220),
    text='metric:N'
).properties(width=500, height=500)

radial_line + radial_labels
